In [2]:

import json
from pathlib import Path
import shutil

import numpy as np
from PIL import Image

from ultralytics.utils import TQDM
from ultralytics.data.split import autosplit
from ultralytics.utils.downloads import download
from ultralytics.utils.ops import xyxy2xywhn


def convert_labels(fname=Path("xView/xView_train.geojson")):
    """Convert xView GeoJSON labels to YOLO format (classes 0-59) and save them as text files."""
    path = fname.parent
    with open(fname, encoding="utf-8") as f:
        print(f"Loading {fname}...")
        data = json.load(f)

    # Make dirs
    labels = path / "labels" / "train"
    shutil.rmtree(labels, ignore_errors=True)
    labels.mkdir(parents=True, exist_ok=True)

    # xView classes 11-94 to 0-59
    xview_class2index = [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 0, 1, 2, -1, 3, -1, 4, 5, 6, 7, 8, -1, 9, 10, 11,
                        12, 13, 14, 15, -1, -1, 16, 17, 18, 19, 20, 21, 22, -1, 23, 24, 25, -1, 26, 27, -1, 28, -1,
                        29, 30, 31, 32, 33, 34, 35, 36, 37, -1, 38, 39, 40, 41, 42, 43, 44, 45, -1, -1, -1, -1, 46,
                        47, 48, 49, -1, 50, 51, -1, 52, -1, -1, -1, 53, 54, -1, 55, -1, -1, 56, -1, 57, -1, 58, 59]

    shapes = {}
    for feature in TQDM(data["features"], desc=f"Converting {fname}"):
        p = feature["properties"]
        if p["bounds_imcoords"]:
            image_id = p["image_id"]
            image_file = path / "train_images" / image_id
            if image_file.exists():  # 1395.tif missing
                try:
                    box = np.array([int(num) for num in p["bounds_imcoords"].split(",")])
                    assert box.shape[0] == 4, f"incorrect box shape {box.shape[0]}"
                    cls = p["type_id"]
                    cls = xview_class2index[int(cls)]  # xView class to 0-59
                    assert 59 >= cls >= 0, f"incorrect class index {cls}"

                    # Write YOLO label
                    if image_id not in shapes:
                        shapes[image_id] = Image.open(image_file).size
                    box = xyxy2xywhn(box[None].astype(float), w=shapes[image_id][0], h=shapes[image_id][1], clip=True)
                    with open((labels / image_id).with_suffix(".txt"), "a", encoding="utf-8") as f:
                        f.write(f"{cls} {' '.join(f'{x:.6f}' for x in box[0])}\n")  # write label.txt
                except Exception as e:
                    print(f"WARNING: skipping one label for {image_file}: {e}")


# Download manually from https://challenge.xviewdataset.org
dir = Path('xview')  # dataset root dir
urls = [
    "https://d307kc0mrhucc3.cloudfront.net/train_labels.zip",  # train labels
    "https://d307kc0mrhucc3.cloudfront.net/train_images.zip",  # 15G, 847 train images
    "https://d307kc0mrhucc3.cloudfront.net/val_images.zip",  # 5G, 282 val images (no labels)
]
download(urls, dir=dir)

# Convert labels
# convert_labels(dir / "xView_train.geojson")

# # Move images
# images = Path(dir / "images")
# images.mkdir(parents=True, exist_ok=True)
# Path(dir / "train_images").rename(dir / "images" / "train")
# Path(dir / "val_images").rename(dir / "images" / "val")

# # Split
# autosplit(dir / "images" / "train")